# ThinkGuard 2단계: Colab 무료 GPU로 한국어 번역

**준비 (Mac에서 한 번만)**: Google Drive의 `내 드라이브`에 `ThinkGuard_model` 폴더를 만들고 아래 파일만 올린다.
```
ThinkGuard_model/
├── scripts/02_translate_ko.py
└── data/joined/train.jsonl, valid.jsonl, test.jsonl
```
**런타임 → 런타임 유형 변경 → T4 GPU** 를 선택한 뒤 위에서부터 차례로 실행한다.

번역 결과와 캐시는 Drive(`data/ko/`)에 바로 저장된다. 세션이 끊기면 **1, 2, 3, 4, 6번 셀만 다시 실행**하면 이어서 번역한다.

## 1. GPU 확인 (Tesla T4가 보여야 함)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/ThinkGuard_model'
%cd {PROJECT}
!ls scripts data/joined

## 3. vLLM 설치 (5~8분)
Colab 기본 Python(3.13)에서는 T4용 vLLM 0.8.5가 설치되지 않는다. 그래서 **uv로 Python 3.12 전용 가상환경**(`/content/vllm_env`)을 만들어 그 안에 설치한다.
번역 스크립트는 Colab 기본 Python으로 실행하고, vLLM 서버만 이 가상환경에서 돌린다.
세션이 끊기면 `/content`가 초기화되니 이 셀도 다시 실행해야 한다.

In [ ]:
!pip install -q uv openai
!uv venv -q --python 3.12 /content/vllm_env
!uv pip install -q --python /content/vllm_env/bin/python vllm==0.8.5 transformers==4.51.3
!/content/vllm_env/bin/python -c "import vllm, sys; print('vllm', vllm.__version__, '/ python', sys.version.split()[0])"

## 4. 번역 모델 서버 띄우기 (모델 다운로드 포함 5~10분)
- 기본: **EXAONE 3.5 7.8B AWQ** (한국어 품질 좋음, 비상업 라이선스 → 학교 프로젝트 OK)
- 대안: `Qwen/Qwen2.5-7B-Instruct-AWQ` (Apache 2.0), 더 빠르게: `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct`

In [ ]:
import os, subprocess, time, urllib.request

MODEL = 'LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ'

log = open('/content/vllm.log', 'w')
server = subprocess.Popen(
    ['/content/vllm_env/bin/python', '-m', 'vllm.entrypoints.openai.api_server', '--model', MODEL,
     '--dtype', 'half', '--max-model-len', '8192', '--gpu-memory-utilization', '0.92',
     '--max-num-seqs', '32', '--trust-remote-code', '--port', '8000'],
    stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, VLLM_USE_V1='0'))

for i in range(120):  # 최대 20분 대기
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5)
        print('서버 준비 완료'); break
    except Exception:
        if server.poll() is not None:
            print('서버가 종료됨. 로그 확인:'); break
        time.sleep(10)
else:
    print('시간 초과. 로그 확인:')
!tail -n 20 /content/vllm.log

## 5. 시험 번역 (split별 5행) → 품질 확인
`message_en`과 `message_ko`를 비교해서 다음을 확인한다.
1. 학생 말투가 자연스러운지
2. 코드와 로그가 보존됐는지
3. 요청 의도와 시도 표현이 살아 있는지

품질이 별로면 4번 셀에서 `MODEL`을 바꿔서 다시 실행한다.

In [ ]:
!python scripts/02_translate_ko.py --backend openai --base-url http://localhost:8000/v1 --model {MODEL} --limit 5 --workers 16 --out-dir data/ko_colab_test

import pandas as pd
pd.set_option('display.max_colwidth', 300)
pd.read_csv('data/ko_colab_test/review_sample.csv')[['message_en', 'message_ko']]

## 6. 전체 번역 (수 시간)
진행 상황이 100개마다 출력된다. 끊겨도 캐시가 Drive에 있으니 다시 실행하면 이어서 한다.

In [ ]:
!python scripts/02_translate_ko.py --backend openai --base-url http://localhost:8000/v1 --model {MODEL} --workers 32 --out-dir data/ko

## 7. 결과 확인
`translation_ok=False`인 행 수를 확인한다. 실패한 항목은 6번 셀을 다시 실행하면 재시도된다.
끝나면 Drive의 `data/ko/` 폴더를 Mac의 `ThinkGuard_model/data/ko/`로 내려받는다.

In [ ]:
!cat data/ko/translate_report.json
!grep -c '"ok": false' data/ko/translation_cache.jsonl || true

---
## (대안) vLLM 설치나 실행이 안 될 때: Ollama
vLLM보다 2~4배 느리지만 설치가 단순하다. 3~4번 대신 아래 셀을 실행하고 5, 6번 셀의 `--base-url`을 `http://localhost:11434/v1`, `--model`을 `exaone3.5:7.8b`로 바꾼다.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, os, time
ollama = subprocess.Popen(['ollama', 'serve'], env=dict(os.environ, OLLAMA_NUM_PARALLEL='8'),
                          stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull exaone3.5:7.8b
MODEL = 'exaone3.5:7.8b'